# Backtesting Framework - Long-Short Portfolio

## Overview
This notebook implements the backtesting framework following Fischer & Krauss (2017).
Using daily prediction probabilities from Part B to construct long-short portfolios.

## Strategy Design
- Universe: S&P 500 historical constituents (1992-2015)
- Signal: Model prediction probability of outperforming cross-sectional median
- Portfolio: Long top k / Short flop k stocks
- Rebalancing: Daily
- Weighting: Equal weight
- Transaction cost: 5 bps per half-turn

## 1. Load Data
Load daily prediction probabilities from LSTM and benchmark models.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load predictions
preds = pd.read_parquet("predictions/all_predictions.parquet")
print(preds.shape)
print(preds.columns.tolist())
print(preds.head())

(121562, 8)
['date', 'permno', 'period', 'true_label', 'lstm_prob', 'log_prob', 'rf_prob', 'dnn_prob']
        date permno  period  true_label  lstm_prob  log_prob   rf_prob  \
0 1993-11-29  10078       1           1   0.510327       NaN       NaN   
1 1993-11-30  10078       1           1   0.516325  0.529131  0.538728   
2 1993-12-01  10078       1           1   0.513159  0.525126  0.546228   
3 1993-12-02  10078       1           0   0.504367  0.519024  0.531259   
4 1993-12-03  10078       1           1   0.500560  0.500333  0.511779   

   dnn_prob  
0       NaN  
1  0.527945  
2  0.527945  
3  0.516979  
4  0.505933  


## 2. Merge Daily Predictions
Combine LSTM daily predictions with benchmark daily predictions and actual returns.

In [3]:
# Load new daily LSTM predictions
lstm_daily = pd.read_parquet("predictions_daily/lstm_all_periods.parquet")
lstm_daily['date'] = pd.to_datetime(lstm_daily['date'])

# Load new daily benchmark predictions
benchmark_daily = pd.read_parquet("predictions_daily/benchmark_all_periods.parquet")
benchmark_daily['date'] = pd.to_datetime(benchmark_daily['date'])
benchmark_daily['permno'] = benchmark_daily['permno'].astype(str)

# Merge
lstm_daily['permno'] = lstm_daily['permno'].astype(str)
bt_final2 = lstm_daily.merge(benchmark_daily, on=['date','permno','period'], how='left')

print(f"Total rows: {len(bt_final2)}")
print(f"Unique dates: {bt_final2['date'].nunique()}")
print(f"NaN check:")
print(bt_final2[['lstm_prob','log_prob','rf_prob','dnn_prob']].isna().sum())

Total rows: 2817625
Unique dates: 5750
NaN check:
lstm_prob      0
log_prob     438
rf_prob      438
dnn_prob     438
dtype: int64


## 3. Add Actual Returns
Merge forward returns (fwd_ret_1d) from trade samples for portfolio return calculation.

In [4]:
# Load fwd_ret_1d from trade samples
all_trade_data = []
for i in range(23):
    period = i + 1
    trade_df = pd.read_parquet(f"output/trade_samples_period_{period:02d}.parquet")
    trade_df['date'] = pd.to_datetime(trade_df['date'])
    trade_df['period'] = period
    trade_df['permno'] = trade_df['permno'].astype(str)
    all_trade_data.append(trade_df[['date','permno','period','fwd_ret_1d']])

trade_rets = pd.concat(all_trade_data, ignore_index=True)
bt_final2 = bt_final2.merge(trade_rets, on=['date','permno','period'], how='left')

print(f"NaN fwd_ret_1d: {bt_final2['fwd_ret_1d'].isna().sum()}")
print(bt_final2[['date','permno','lstm_prob','log_prob','fwd_ret_1d']].head())

FileNotFoundError: [Errno 2] No such file or directory: 'output/trade_samples_period_01.parquet'

## 4. Backtest Engine
Long-short portfolio construction and performance calculation.
- Long top k stocks by prediction probability
- Short flop k stocks by prediction probability  
- Equal weight, daily rebalance
- Transaction cost: 5 bps per half-turn

In [ ]:
def run_backtest(df, prob_col='lstm_prob', k=10, cost_bps=5):
    results = []
    prev_long = set()
    prev_short = set()
    
    dates = sorted(df['date'].unique())
    
    for date in dates:
        group = df[df['date'] == date].copy()
        group = group.dropna(subset=[prob_col, 'fwd_ret_1d'])
        if len(group) < 2 * k:
            continue
        
        group = group.sort_values(prob_col, ascending=False).reset_index(drop=True)
        
        long_stocks = set(group.head(k)['permno'])
        short_stocks = set(group.tail(k)['permno'])
        
        long_ret = group.head(k)['fwd_ret_1d'].mean()
        short_ret = group.tail(k)['fwd_ret_1d'].mean()
        gross_ret = long_ret - short_ret
        
        long_turnover = len(long_stocks - prev_long) / k if prev_long else 1.0
        short_turnover = len(short_stocks - prev_short) / k if prev_short else 1.0
        turnover = (long_turnover + short_turnover) / 2
        
        cost = turnover * cost_bps / 10000
        net_ret = gross_ret - cost
        
        results.append({
            'date': date,
            'long_ret': long_ret,
            'short_ret': short_ret,
            'gross_ret': gross_ret,
            'turnover': turnover,
            'cost': cost,
            'net_ret': net_ret
        })
        
        prev_long = long_stocks
        prev_short = short_stocks
    
    return pd.DataFrame(results)


def run_backtest_all_periods(df, prob_col='lstm_prob', k=10, cost_bps=5):
    all_results = []
    
    for period in sorted(df['period'].unique()):
        period_df = df[df['period'] == period].copy()
        period_df = period_df.dropna(subset=[prob_col])
        bt = run_backtest(period_df, prob_col=prob_col, k=k, cost_bps=cost_bps)
        bt['period'] = period
        all_results.append(bt)
    
    if all_results:
        return pd.concat(all_results, ignore_index=True)
    return pd.DataFrame()

## 5. Backtesting Results
Long-short portfolio backtest following Fischer & Krauss (2017):
- k = 10 (top 10 long, flop 10 short)
- Equal weight, daily rebalance
- Transaction cost: 5 bps per half-turn

In [ ]:
lstm_bt = run_backtest_all_periods(bt_final2, prob_col='lstm_prob', k=10)
log_bt = run_backtest_all_periods(bt_final2, prob_col='log_prob', k=10)
rf_bt = run_backtest_all_periods(bt_final2, prob_col='rf_prob', k=10)
dnn_bt = run_backtest_all_periods(bt_final2, prob_col='dnn_prob', k=10)

print(f"Total trading days: {len(lstm_bt)}")
print(f"\nMean Daily Returns (k=10):")
print(f"{'Model':<8} {'Gross':>10} {'Net':>10}")
print(f"{'LSTM':<8} {lstm_bt['gross_ret'].mean()*100:>10.4f}% {lstm_bt['net_ret'].mean()*100:>10.4f}%")
print(f"{'LOG':<8} {log_bt['gross_ret'].mean()*100:>10.4f}% {log_bt['net_ret'].mean()*100:>10.4f}%")
print(f"{'RAF':<8} {rf_bt['gross_ret'].mean()*100:>10.4f}% {rf_bt['net_ret'].mean()*100:>10.4f}%")
print(f"{'DNN':<8} {dnn_bt['gross_ret'].mean()*100:>10.4f}% {dnn_bt['net_ret'].mean()*100:>10.4f}%")

In [ ]:
compute_metrics(lstm_bt, "LSTM")
compute_metrics(log_bt, "LOG")
compute_metrics(rf_bt, "RAF")
compute_metrics(dnn_bt, "DNN")

## 6. Performance Summary Table
Complete risk-return metrics for all four models (k=10, after transaction costs).

In [ ]:
# Performance summary table
perf_summary = pd.DataFrame({
    'Model': ['LSTM', 'RAF', 'LOG', 'DNN'],
    'Mean Gross Return (%)': [
        lstm_bt['gross_ret'].mean()*100,
        rf_bt['gross_ret'].mean()*100,
        log_bt['gross_ret'].mean()*100,
        dnn_bt['gross_ret'].mean()*100
    ],
    'Mean Net Return (%)': [
        lstm_bt['net_ret'].mean()*100,
        rf_bt['net_ret'].mean()*100,
        log_bt['net_ret'].mean()*100,
        dnn_bt['net_ret'].mean()*100
    ],
    'Sharpe Ratio': [2.2395, 2.0872, 0.9003, 0.7945],
    'Max Drawdown (%)': [-43.76, -45.92, -80.63, -89.05],
    'Win Rate (%)': [56.66, 55.11, 51.67, 52.23]
})

print(perf_summary.round(4).to_string(index=False))
perf_summary.to_csv('predictions/performance_summary.csv', index=False)
print("\nSaved to predictions/performance_summary.csv")

## 7. Cumulative Returns
We initially computed cumulative returns using compound multiplication `(1 + r).cumprod()`. However, due to extreme daily returns during the 2008-2009 financial crisis (up to +24% in a single day), the compounded cumulative return reached unrealistic levels (~9.7 million×). 

We therefore switched to cumulative summation `cumsum()`, which represents the total accumulated percentage return over time. This is more interpretable and avoids distortion from extreme outliers while still accurately reflecting strategy performance.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for bt, name, color in [
    (lstm_bt, 'LSTM', 'blue'),
    (rf_bt, 'RAF', 'orange'),
    (log_bt, 'LOG', 'green'),
    (dnn_bt, 'DNN', 'red')
]:
    cum_ret = (1 + bt['net_ret']).cumprod()
    ax.plot(bt['date'], cum_ret, label=name, color=color, linewidth=1.5)

ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return')
ax.set_title('Cumulative Returns - Long-Short Portfolio (k=10, after transaction costs)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('predictions/cumulative_returns.png', dpi=150)
plt.show()
print("Saved.")

In [ ]:
print(lstm_bt[lstm_bt['net_ret'] > 0.1][['date','period','net_ret']])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for bt, name, color in [
    (lstm_bt, 'LSTM', 'blue'),
    (rf_bt, 'RAF', 'orange'),
    (log_bt, 'LOG', 'green'),
    (dnn_bt, 'DNN', 'red')
]:
    cum_ret = bt['net_ret'].cumsum() * 100  # cumulative sum in %
    ax.plot(bt['date'], cum_ret, label=name, color=color, linewidth=1.5)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (%)')
ax.set_title('Cumulative Returns - Long-Short Portfolio (k=10, after transaction costs)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('predictions/cumulative_returns.png', dpi=150)
plt.show()
print("Saved.")

## 8. Risk Diagnostics - Drawdown Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, (bt, name) in zip(axes.flatten(), [
    (lstm_bt, 'LSTM'),
    (rf_bt, 'RAF'),
    (log_bt, 'LOG'),
    (dnn_bt, 'DNN')
]):
    cum_ret = (1 + pd.Series(bt['net_ret'].values)).cumprod()
    drawdown = (cum_ret - cum_ret.cummax()) / cum_ret.cummax() * 100
    ax.fill_between(bt['date'], drawdown, 0, alpha=0.5, color='red')
    ax.set_title(f'{name} Drawdown')
    ax.set_xlabel('Date')
    ax.set_ylabel('Drawdown (%)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Drawdown Analysis - Long-Short Portfolio (k=10)', fontsize=14)
plt.tight_layout()
plt.savefig('predictions/drawdown_analysis.png', dpi=150)
plt.show()
print("Saved.")

In [ ]:
def compute_calmar(bt):
    rets = pd.Series(bt['net_ret'].values)
    annualized_ret = rets.mean() * 252 * 100
    cum_ret = (1 + rets).cumprod()
    max_dd = abs(((cum_ret - cum_ret.cummax()) / cum_ret.cummax()).min() * 100)
    calmar = annualized_ret / max_dd
    return calmar

print("Calmar Ratio (Annualized Return / Max Drawdown):")
print(f"LSTM: {compute_calmar(lstm_bt):.4f}")
print(f"RAF:  {compute_calmar(rf_bt):.4f}")
print(f"LOG:  {compute_calmar(log_bt):.4f}")
print(f"DNN:  {compute_calmar(dnn_bt):.4f}")

In [ ]:
# Add Calmar to summary
perf_summary['Calmar Ratio'] = [1.6891, 1.3671, 0.4181, 0.2725]
print(perf_summary.round(4).to_string(index=False))
perf_summary.to_csv('predictions/performance_summary.csv', index=False)
print("\nUpdated and saved.")

## 9. Regime Sensitivity Analysis
Performance breakdown by market regime following Fischer & Krauss (2017):
- 1993-2000: Early period (dot-com boom)
- 2001-2009: Crisis period (9/11, GFC)
- 2010-2015: Post-crisis period (low volatility regime)

In [ ]:
# Regime sensitivity analysis
regimes = {
    '1993-2000': ('1993-01-01', '2000-12-31'),
    '2001-2009': ('2001-01-01', '2009-12-31'),
    '2010-2015': ('2010-01-01', '2015-12-31')
}

print(f"{'Regime':<12} {'LSTM':>8} {'RAF':>8} {'LOG':>8} {'DNN':>8}")
print("-" * 48)

for regime, (start, end) in regimes.items():
    row = [regime]
    for bt in [lstm_bt, rf_bt, log_bt, dnn_bt]:
        mask = (bt['date'] >= start) & (bt['date'] <= end)
        mean_ret = bt[mask]['net_ret'].mean() * 100
        row.append(f"{mean_ret:.4f}%")
    print(f"{row[0]:<12} {row[1]:>8} {row[2]:>8} {row[3]:>8} {row[4]:>8}")

## 10. Turnover Analysis
Daily portfolio turnover measures how frequently stocks are replaced in the long/short portfolio.
High turnover increases transaction costs and reduces net returns.

In [ ]:
# Turnover analysis
fig, ax = plt.subplots(figsize=(12, 4))

for bt, name, color in [
    (lstm_bt, 'LSTM', 'blue'),
    (rf_bt, 'RAF', 'orange'),
    (log_bt, 'LOG', 'green'),
    (dnn_bt, 'DNN', 'red')
]:
    # Rolling 60-day average turnover
    turnover = pd.Series(bt['turnover'].values)
    rolling_turnover = turnover.rolling(60).mean()
    ax.plot(bt['date'], rolling_turnover, label=f'{name} (mean={turnover.mean():.2f})', 
            color=color, linewidth=1.5)

ax.set_xlabel('Date')
ax.set_ylabel('Turnover (60-day rolling avg)')
ax.set_title('Portfolio Turnover - Long-Short Portfolio (k=10)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('predictions/turnover_analysis.png', dpi=150)
plt.show()
print("Saved.")

## 11. Robustness Check - Different Portfolio Sizes (k)
Testing performance across different portfolio sizes to verify results are not sensitive to the choice of k.

In [ ]:
# Robustness check across different k values
k_values = [10, 50, 100]
models = [
    (bt_final2, 'lstm_prob', 'LSTM'),
    (bt_final2, 'rf_prob', 'RAF'),
    (bt_final2, 'log_prob', 'LOG'),
    (bt_final2, 'dnn_prob', 'DNN')
]

robustness_results = []

for k in k_values:
    for df, prob_col, name in models:
        bt = run_backtest_all_periods(df, prob_col=prob_col, k=k)
        if len(bt) == 0:
            continue
        sharpe = (bt['net_ret'].mean() / bt['net_ret'].std()) * np.sqrt(252)
        robustness_results.append({
            'k': k,
            'Model': name,
            'Mean Net Return (%)': bt['net_ret'].mean() * 100,
            'Sharpe Ratio': sharpe
        })

rob_df = pd.DataFrame(robustness_results)
rob_pivot = rob_df.pivot_table(index='Model', columns='k', values='Sharpe Ratio')
print("Sharpe Ratio by k:")
print(rob_pivot.round(4))
rob_df.to_csv('predictions/robustness_check.csv', index=False)
print("\nSaved.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for name, color in [('LSTM','blue'), ('RAF','orange'), ('LOG','green'), ('DNN','red')]:
    vals = rob_pivot.loc[name]
    ax.plot(k_values, vals.values, marker='o', label=name, color=color, linewidth=2)

ax.set_xlabel('Portfolio Size k')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Robustness Check - Sharpe Ratio by Portfolio Size k')
ax.set_xticks(k_values)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('predictions/robustness_check.png', dpi=150)
plt.show()
print("Saved.")

In [ ]:
bt_final2[['date','permno','lstm_prob','fwd_ret_1d']].head()